# (Generalized) coordination numbers in pySNOW

Coordination is a general and simple property of atoms. In the most basic case, coordination is a count of the number of neighbours of an atom, as we define the coordination number of the atom $i$ as 
$$
CN(i) = \sum_{j\in\mathcal{N}(i)}1 = n_i
$$
where $j$ runs over the $n_i$ atoms in the neighbourhood of the atom $i$: $\mathcal{N}(i)=\{\mathbf{r}_j:|\mathbf{r}_j-\mathbf{r}_i|<r_{cut}\}$, where $\mathbf{r}_\alpha$ is the position vector of the atom $\alpha$.

Naturally, we need to choose a cutoff $r_{cut}$ to define which atoms are neighbours and which are not. See our tutorial on PDDF for a discussion on this.

Some more complete and insightful information can be obtained from so-called generalized coordination numbers (GCNs), which take into account the coordination of neighbours as well to define the coordination of a site. In particular, GCNs are useful as descriptors of catalytic activity [1]. GCNs can be defined for atoms (_atop_ GCN - aGCN) or for sites generally considered as adsorption sites in atomistic systems (bridge - bGCN, three-fold hollow - thGCN, four-fold hollow - fhGCN) [2].

In the atop version:
$$
aGCN(i) = \frac{1}{CN_{max}} \sum_{j=1}^{n_i} CN(j)
$$
where $j$ runs over the $n_i$ neighbours of the atom $i$, and $CN_{max}$ is the coordination of atoms of the same species in the bulk (e.g. $CN_{max}=12$ for FCC materials).

For bridge and hollow sites, the sum over coordination of neighbours is performed over all the $n_i$ unique neighbours of the pair/triplet/fourplet $i$, and the max coordination is adjusted according to the number of corresponding neighbours in the bulk (in the case of FCC materials, $CN_{max}=18$ for bridge sites, $CN_{max}=22$ for three-fold hollow sites, and $CN_{max}=26$ for four-fold hollow sites).

See how to compute gcns in pySNOW:

In [1]:
from snow.io.xyz import read_xyz, write_xyz

#read some example data
file = 'tutorial_structures/Au13_ih.xyz'

el, coords = read_xyz(file)

In [2]:
#compute the coordination number and atop generalized coordination number
from snow.descriptors.coordination import coordination_number, agcn_calculator

alat   = 4.099
cutoff = 0.89*alat

cns   = coordination_number(coords, cutoff)
sites, agcns = agcn_calculator(coords, cutoff)

#print the first five computed elements
print('                   coordination number of the first five atoms:',cns[:5])
print('(atop) generalized coordination number of the first five atoms:',agcns[:5])
#print the coordinates of the site corresponding to the first computed value:
print('                                         coordinates of site 0:',sites[0])
#here, this is just the position of the first atom. for bridge/hollow sites, the site position is in between two or more atoms.

                   coordination number of the first five atoms: [12.  6.  6.  6.  6.]
(atop) generalized coordination number of the first five atoms: [6.  3.5 3.5 3.5 3.5]
                                         coordinates of site 0: [0. 0. 0.]


In [3]:
#compute some other types of gcns
from snow.descriptors.coordination import bridge_gcn, three_hollow_gcn, four_hollow_gcn
 

#bridge sites are only considered between atoms on the surface of your structure.
#An atom is considered on the surface if its coordination is < thr_cn
#use phantom=True to have your function return the pairs of indexes of atoms making up bridge sites as well
bridge_sites, bridge_pairs, bgcns = bridge_gcn(coords, cutoff, thr_cn = 10)

th_sites, th_gcns = three_hollow_gcn(coords, cutoff, thr_cn = 10)
fh_sites, fh_gcns = four_hollow_gcn(coords, cutoff, thr_cn = 10)

[==================================================] 100.00%
Done three hollow
[=================================================-] 99.88%
Done four hollow


Strained structures have different adsorption energies and catalytic activities compared to relaxed ones. To take this into account, one can consider a strain-aware version of the GCNs we discussed [3]. Here, strained is defined as 

$$
S = \frac{d_{ij}-d_{bulk}}{d_{bulk}}
$$
where $d_{ij}$ is the distance between atoms $i$ and $j$, and $d_{bulk}$ is the corresponding nearest neighbours distance in the unstrained bulk. A system-wide measure of the strain can be obtained considering the average distance between pairs of nearest neighbours $\bar{d}_{ij}$. This way, compressive strain is negative and tensile strain is positive, while $S=0$ indicates an unstrained/relaxed structure. 

The strain-aware version of GCN is then defined as
$$
GCN^*(i) = \frac{1}{CN_{max}}\sum_{j=1}^{n_i}\sum_{k=1}^{n_j} \frac{d_{bulk}}{d_{jk}} \approx \frac{1}{1+S}GCN(i)
$$

In [4]:
#to compute strained GCNs simply use strained=True as an argument in the calculator function. 
#If this is the case, you should also pass the bulk nearest neighbours distance as the argument d_bulk to the function.

You might want to export your computed gcns, e.g., for visualization with OVITO. To do this, you can write fictitious atoms (e.g., Hydrogen atoms if you do not have any Hydrogen in your system) to store data about bridge/hollow sites and their generalized coordination.

In [5]:
import numpy as np

fake_specie = 'H'

#write bridge sites as H atoms
#with info about their generalized coordination 
#as an extra-info array
fake_elements = el + [fake_specie]*len(bridge_sites)

#for now, we need to convert bridge_sites (a list) to a np.ndarray. We'll fix this in the future
bridge_sites_array = np.asarray(bridge_sites)
print(bridge_sites_array.shape)
print(coords.shape)
print(bridge_sites_array[0])
fake_coords = np.vstack((coords, bridge_sites_array))

extra_properties = np.concatenate((np.zeros(len(el)), bgcns))

write_xyz('tutorial_structures/Au13_ih_bgcns.xyz', fake_elements, fake_coords, additional_data=extra_properties)

(30, 3)
(13, 3)
[ 0.758365 -1.22706  -1.985425]


### References

Some work from F. Calle-Vallejo and coworkers:

[1 - Introduction to generalized coordination numbers](https://doi.org/10.1002/advs.202207644)

[2 -Bridge and hollow coordination numbers](https://doi.org/10.1021/acsomega.7b01126)

[3 - Strained generalized coordination numbers](https://chemistry-europe.onlinelibrary.wiley.com/doi/full/10.1002/cssc.201800569)